# Re-annotation — agreement evaluation, both rounds

Two annotators, two rounds, evaluated **separately**. Nothing is pooled across
rounds: they were drawn from different populations and answer different
questions.

| Round | Items | Population | Question |
|---|---|---|---|
| 1 | 650 | corpus sample, proportional + news top-up | how well do the published labels hold up |
| 2 | 250 | locked test set, proportional | how far can any classifier go on this test set |

Round 1 also contains 95 context items that are never evaluated. They kept the
batch composition identical for both annotators and carry no claim.

Three labels: Negative / Neutral / Positive. Cohen's kappa is unweighted.

## 1 · Config and load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd, numpy as np
from math import sqrt
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the v2_heldout "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/v2_heldout'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)

from config import ANNOTATION, TEST_EVAL

ANN = ANNOTATION
R1, R2 = ANN + "/round1", ANN + "/round2"
TEST_RESULTS = os.path.join(TEST_EVAL, "test_results.csv")

CHOICE_COL = "sentiment"
SEED       = 42
LABELS3    = ["Negative", "Neutral", "Positive"]
L2I        = {n: i for i, n in enumerate(LABELS3)}


def clean_choice(x):
    return str(x).strip().strip("[]' ").capitalize()


def load_round(key_path, export_a, export_b, name):
    if not os.path.exists(key_path):
        print(f"{name}: key missing - skipped")
        return None
    key = pd.read_csv(key_path, encoding="utf-8-sig")

    frames = [key]
    for path, suffix in ((export_a, "A"), (export_b, "B")):
        if not os.path.exists(path):
            print(f"{name}: export {suffix} missing - skipped")
            return None
        e = pd.read_csv(path, encoding="utf-8-sig")
        cols = {"id": e["id"], f"label_{suffix}": e[CHOICE_COL].map(clean_choice)}
        if "note" in e.columns:
            cols[f"note_{suffix}"] = e["note"]
        frames.append(pd.DataFrame(cols))

    df = frames[0]
    for f in frames[1:]:
        df = df.merge(f, on="id", how="inner")

    assert len(df) == len(key), f"{name}: ids do not line up ({len(df)}/{len(key)})"
    for s in ("A", "B"):
        bad = set(df[f"label_{s}"]) - set(LABELS3)
        assert not bad, f"{name}: unexpected labels from {s}: {bad}"
    print(f"{name}: {len(df)} items joined")
    return df


d1 = load_round(f"{R1}/round1_key_PRIVATE.csv",
                f"{R1}/round1_export_annotatorA.csv",
                f"{R1}/round1_export_annotatorB.csv", "round 1")
d2 = load_round(f"{R2}/round2_key_PRIVATE.csv",
                f"{R2}/round2_export_annotatorA.csv",
                f"{R2}/round2_export_annotatorB.csv", "round 2")

if d1 is not None:
    ev1 = d1[d1["evaluated"]].copy()
    print(f"  round 1 evaluated: {len(ev1)}  (context excluded: {len(d1) - len(ev1)})")

## 2 · Statistical helpers

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = (z*sqrt(p*(1-p)/n + z*z/(4*n*n))) / d
    return p, max(0, c-h), min(1, c+h)


def kappa_boot(y1, y2, B=2000, seed=SEED):
    y1, y2 = np.asarray(y1), np.asarray(y2)
    k = cohen_kappa_score(y1, y2)
    rng = np.random.default_rng(seed); n = len(y1); vals = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(set(y1[idx])) < 2 or len(set(y2[idx])) < 2:
            continue
        vals.append(cohen_kappa_score(y1[idx], y2[idx]))
    lo, hi = (np.nanpercentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan))
    return k, lo, hi


def pabak3(po):
    return (3*po - 1) / 2


def compare(sub, c1, c2, title, n1, n2, show_cm=True):
    y1 = sub[c1].map(L2I).values
    y2 = sub[c2].map(L2I).values
    po = float((y1 == y2).mean()) if len(sub) else np.nan
    k, klo, khi = kappa_boot(y1, y2) if len(sub) >= 10 else (np.nan,)*3
    err, elo, ehi = wilson(int((y1 != y2).sum()), len(sub))
    print(f"===== {title} =====")
    print(f"  items                     : {len(sub)}")
    print(f"  raw agreement  Po         : {po:6.1%}")
    print(f"  disagreement rate         : {err:6.1%}  (95% CI {elo:.1%}-{ehi:.1%})")
    print(f"  Cohen's kappa (unweighted): {k:6.3f}  (95% CI {klo:.3f}-{khi:.3f})")
    print(f"  PABAK                     : {pabak3(po):6.3f}")
    if show_cm:
        cm = confusion_matrix(y1, y2, labels=[0, 1, 2])
        print(f"  confusion (rows={n1}, cols={n2})  order Neg/Neu/Pos:")
        print(pd.DataFrame(cm, index=LABELS3, columns=LABELS3)
              .to_string().replace("\n", "\n  "))
    print()
    return dict(title=title, n=len(sub), Po=po, err=err,
                kappa=k, kappa_lo=klo, kappa_hi=khi, pabak=pabak3(po))

## 3 · Round 1 — corpus sample

The representative arm is proportional to label x category and carries the
overall figure. The news slice adds the top-up items.

In [ ]:
rows1 = []
if d1 is not None:
    rep  = ev1[ev1.arm_eval == "representative"]
    news = ev1[ev1.category == "news"]
    print(f"representative {len(rep)} | news {len(news)} | all evaluated {len(ev1)}\n")

    rows1.append(compare(rep,  "gold_name", "label_A", "R1 A vs GOLD (representative)", "gold", "A"))
    rows1.append(compare(rep,  "gold_name", "label_B", "R1 B vs GOLD (representative)", "gold", "B"))
    rows1.append(compare(rep,  "label_A",   "label_B", "R1 A vs B    (representative)", "A", "B"))
    rows1.append(compare(news, "gold_name", "label_A", "R1 A vs GOLD (news)", "gold", "A", show_cm=False))
    rows1.append(compare(news, "gold_name", "label_B", "R1 B vs GOLD (news)", "gold", "B", show_cm=False))
    rows1.append(compare(news, "label_A",   "label_B", "R1 A vs B    (news)", "A", "B", show_cm=False))
    rows1.append(compare(ev1,  "label_A",   "label_B", "R1 A vs B    (all 650)", "A", "B", show_cm=False))

## 4 · Round 1 — three-way pattern

`both_agree_vs_gold` is the interesting cell: both annotators independently
chose the same label and it is not the published one. That is evidence about the
gold label, obtained without any model.

In [ ]:
def three_way(ev, label="round 1"):
    ev = ev.copy()
    ev["A_eq_gold"] = ev["label_A"] == ev["gold_name"]
    ev["B_eq_gold"] = ev["label_B"] == ev["gold_name"]
    ev["A_eq_B"]    = ev["label_A"] == ev["label_B"]

    def pat(r):
        if r.A_eq_gold and r.B_eq_gold:     return "all_three_agree"
        if r.A_eq_B and not r.A_eq_gold:    return "both_agree_vs_gold"
        if r.A_eq_gold:                     return "only_B_differs"
        if r.B_eq_gold:                     return "only_A_differs"
        return "all_three_differ"

    ev["pattern"] = ev.apply(pat, axis=1)
    print(f"Three-way pattern - {label}, n={len(ev)}\n")
    for name, n in ev["pattern"].value_counts().items():
        p, lo, hi = wilson(n, len(ev))
        print(f"  {name:20s} {n:4d}  ({p:5.1%}, 95% CI {lo:.1%}-{hi:.1%})")

    bg = ev[ev["pattern"] == "both_agree_vs_gold"]
    if len(bg):
        print(f"\n  both annotators agree against gold: {len(bg)}")
        print("  " + pd.crosstab(bg["gold_name"], bg["label_A"])
              .rename_axis(index="gold", columns="both")
              .to_string().replace("\n", "\n  "))
    if "category" in ev.columns:
        print("\n  by category:")
        print("  " + pd.crosstab(ev["category"], ev["pattern"])
              .to_string().replace("\n", "\n  "))
    return ev


if d1 is not None:
    ev1 = three_way(ev1, "round 1, evaluated items")

## 5 · Round 1 — free-text notes

Items either annotator flagged are a guideline-driven difficulty signal that
owes nothing to any model.

In [ ]:
if d1 is not None:
    note_cols = [c for c in ev1.columns if c.startswith("note_")]
    if note_cols:
        ev1["flagged"] = ev1[note_cols].notna().any(axis=1)
        print(f"Items with at least one note: {int(ev1['flagged'].sum())} of {len(ev1)}")
        for c in note_cols:
            s = ev1[c].dropna().astype(str).str.strip().str.lower()
            if len(s):
                print(f"\n{c}: {len(s)} notes")
                print(s.value_counts().head(10).to_string())

        dis = ev1[~ev1["A_eq_B"]]
        if len(dis):
            pf, flo, fhi = wilson(int(dis["flagged"].sum()), len(dis))
            ag = ev1[ev1["A_eq_B"]]
            pa, alo, ahi = wilson(int(ag["flagged"].sum()), len(ag))
            print(f"\nflagged among disagreements: {pf:5.1%} (95% CI {flo:.1%}-{fhi:.1%}), n={len(dis)}")
            print(f"flagged among agreements   : {pa:5.1%} (95% CI {alo:.1%}-{ahi:.1%}), n={len(ag)}")
    else:
        print("no note column in the exports")

## 6 · Round 2 — the test-set ceiling

Agreement between two trained annotators on a proportional random sample of the
locked test set. No model was involved in drawing it. This is an upper bound on
what any classifier can reach on this test set under these guidelines.

In [ ]:
rows2 = []
if d2 is not None:
    rows2.append(compare(d2, "label_A", "label_B", "R2 A vs B (ceiling)", "A", "B"))
    rows2.append(compare(d2, "gold_name", "label_A", "R2 A vs GOLD", "gold", "A", show_cm=False))
    rows2.append(compare(d2, "gold_name", "label_B", "R2 B vs GOLD", "gold", "B", show_cm=False))

    acc_A = float((d2["label_A"] == d2["gold_name"]).mean())
    acc_B = float((d2["label_B"] == d2["gold_name"]).mean())
    for nm, a in (("A", acc_A), ("B", acc_B)):
        p, lo, hi = wilson(int(round(a*len(d2))), len(d2))
        print(f"human accuracy vs gold, {nm}: {p:.1%}  (95% CI {lo:.1%}-{hi:.1%})")

    print("\nBy category (agreement between A and B):")
    for cat, sub in d2.groupby("category"):
        if len(sub) < 20:
            print(f"  {cat:11s} n={len(sub):3d}  too small")
            continue
        k, lo, hi = kappa_boot(sub["label_A"].map(L2I).values,
                               sub["label_B"].map(L2I).values)
        print(f"  {cat:11s} n={len(sub):3d}  kappa={k:.3f}  (95% CI {lo:.3f}-{hi:.3f})")

    d2 = three_way(d2, "round 2, test ceiling")

## 7 · Ceiling against the models

Runs only once `test_evaluation_v2` has produced `test_results.csv`. Places the
architecture differences next to the human agreement on the same test set.

In [ ]:
if d2 is not None and os.path.exists(TEST_RESULTS):
    res = pd.read_csv(TEST_RESULTS, index_col=0)
    print("Model performance on the full test set:")
    print(res.round(4).to_string())

    # column name follows the summary written by test_evaluation_v2
    spread = res["test_ensemble_f1"].max() - res["test_ensemble_f1"].min()
    print(f"\nspread between architectures : {spread:.4f} macro-F1")
    print(f"human accuracy vs gold (A/B) : {acc_A:.1%} / {acc_B:.1%}  on n={len(d2)}")
    print(f"human agreement A vs B       : "
          f"{(d2['label_A'] == d2['label_B']).mean():.1%} raw, kappa={rows2[0]['kappa']:.3f}")
    print("\nIf the architecture spread is small relative to the room the human"
          "\nceiling leaves open, the bottleneck is the task, not the model.")
elif d2 is not None:
    print("test_results.csv not present yet - run test_evaluation_v2 first.")

## 8 · Summary

In [ ]:
def to_frame(rows, label):
    if not rows:
        return None
    s = pd.DataFrame(rows).set_index("title")
    s = s[["n", "Po", "err", "kappa", "kappa_lo", "kappa_hi", "pabak"]]
    s.columns = ["n", "Po", "disagree_rate", "kappa", "k_lo", "k_hi", "PABAK"]
    s.insert(0, "round", label)
    return s

parts = [f for f in (to_frame(rows1, 1), to_frame(rows2, 2)) if f is not None]
if parts:
    summary = pd.concat(parts)
    display(summary.round(3))
    out = f"{ANN}/agreement_summary_both_rounds.csv"
    summary.round(4).to_csv(out, encoding="utf-8")
    print("saved", out)

    if d1 is not None:
        ev1.to_csv(f"{R1}/round1_merged_labels.csv", index=False, encoding="utf-8")
    if d2 is not None:
        d2.to_csv(f"{R2}/round2_merged_labels.csv", index=False, encoding="utf-8")
    print("merged label files written")

## Reading the result

**Round 1.** Compare `A vs B` against `A vs GOLD` and `B vs GOLD` on the
representative arm. If the annotators reproduce each other but not the published
labels, that points at the gold. If all three sit at a similar level, the
disagreement is spread evenly, which is what genuine task ambiguity looks like.
If `A vs B` is as low as either against gold, the guidelines themselves leave too
much open.

**Round 2.** The kappa between A and B is the ceiling. Reported next to the test
macro-F1 of the three architectures, it says whether the difference between them
sits inside or outside the range the task definition leaves open.

Two things not to do with round 2: do not correct the test gold labels, and do
not select which test items to report on. It measures the test set; it does not
revise it.

Report the two rounds separately. They were drawn from different populations —
round 1 straddles the development pool and the test set and includes a news
top-up, round 2 is a clean proportional sample of the test set alone.